<a href="https://colab.research.google.com/github/gitmystuff/DTSC5502/blob/main/Module_10-GLMs/Wikinator_Complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wikinator I



## Document Object Model (DOM)

The Document Object Model (DOM) is a cross-platform and language-independent interface that treats an HTML or XML document as a tree structure wherein each node is an object representing a part of the document. The DOM represents a document with a logical tree. Each branch of the tree ends in a node, and each node contains objects. DOM methods allow programmatic access to the tree; with them one can change the structure, style or content of a document. Nodes can have event handlers (also known as event listeners) attached to them. Once an event is triggered, the event handlers get executed.

https://en.wikipedia.org/wiki/Document_Object_Model

## Wikipedia API

Application Program Interface

If you intend to do any scraping projects or automated requests, consider alternatives such as Pywikipediabot or MediaWiki API, which has other superior features.

* wikipedia.search('keywords', results=2)
* wikipedia.suggest('keyword')
* wikipedia.summary('keywords', sentences=2)
* wikipedia.page('keywords')
* wikipedia.page('keywords').content
* wikipedia.page('keywords').references
* wikipedia.page('keywords').title
* wikipedia.page('keywords').url
* wikipedia.page('keywords').categories
* wikipedia.page('keywords').content
* wikipedia.page('keywords').links
* wikipedia.geosearch(33.2075, 97.1526)
* wikipedia.set_lang('hi')
* wikipedia.languages()
* wikipedia.page('keywords').images[0]
* wikipedia.page('keywords').html()

In [ ]:
pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11679 sha256=9a6e02c05009dc68620adae1aa60a0f1cd7bd6e1ecac1397f2b3081ab1d1258c
  Stored in directory: /root/.cache/pip/wheels/5e/b6/c5/93f3dec388ae76edc830cb42901bb0232504dfc0df02fc50de
Successfully built wikipedia


In [ ]:
# https://kleiber.me/blog/2017/07/22/tutorial-lda-wikipedia/
import pandas as pd
import random
import wikipedia

# titles = wikipedia.random(5)

# get 5 Wikipedia page titles based on keywords
titles = []
keywords = ['ultranationalism', 'religion', 'religious facism', 'state religion', 'deifying rulers']
for key in keywords:
    title = wikipedia.search(key, results=5)
    titles.append(title[0])

data = []

for title in titles:
    # disambiguous error fix
    try:
        url_title = title.strip().replace(' ', '_')
        url = f'https://en.wikipedia.org/wiki/{url_title}' # left alt, shift, down to duplicate line
        # data.append([title, url, wikipedia.page(title, auto_suggest=False).content, wikipedia.summary(title, auto_suggest=False, sentences=15)])
        data.append([title, url])
    except wikipedia.exceptions.DisambiguationError as e:
        s = random.choice(e.options)
        data.append([title, wikipedia.page(s).content,  wikipedia.summary(title, auto_suggest=False, sentences=15)])

pages = pd.DataFrame(data, columns=['title', 'url'])
pages.head()

,title,url
0,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism
1,Religion,https://en.wikipedia.org/wiki/Religion
2,Fascism,https://en.wikipedia.org/wiki/Fascism
3,State religion,https://en.wikipedia.org/wiki/State_religion
4,Apotheosis,https://en.wikipedia.org/wiki/Apotheosis


## Beautiful Soup

In [ ]:
import requests
from bs4 import BeautifulSoup
import pprint
import re

CLEANR = re.compile('<.*?>')
def cleanhtml(raw_html):
  cleantext = re.sub(CLEANR, '', raw_html)
  return cleantext

data = []
def get_subs(row):
  title = row['title']
  url = row['url']

  response = requests.get(url)
  html_content = response.text

  soup = BeautifulSoup(html_content, 'html.parser')
  h2_tags = soup.find_all('h2')

  # Get the text at the beginning of the article, before the first h2 tag
  content_div = soup.find('div', class_='mw-parser-output')
  text_before_first_h2 = ""
  for element in content_div.children:
      if element.name == 'p':
          text_before_first_h2 += element.get_text(strip=False) + "\n"
      elif element.name == 'h2':
          break

  text_before_first_h2 = text_before_first_h2.replace("\n", " ")  # Replace newlines with spaces
  text_before_first_h2 = re.sub(r"\[\d+\]", "", text_before_first_h2)

  data.append([title, url, 'Intro', cleanhtml(text_before_first_h2)])

  # Extract text between h2 tags
  for i in range(len(h2_tags) - 1):
      h2_tag = h2_tags[i]
      next_h2_tag = h2_tags[i + 1]

      elements_between_h2s = h2_tag.find_all_next(string=True, limit=next_h2_tag.sourcepos)
      text_between_h2s = ""
      for element in elements_between_h2s:
          if element.parent.name == 'p':
              # Get the text content of the <p> tag, including the text within <a> tags
              for content in element.parent.contents:
                  if isinstance(content, str):
                      text_between_h2s += content
                  elif content.name == 'a':
                      text_between_h2s += content.text
              # text_between_h2s += "\n"  # Add a newline after each <p>

      # print(f"Text between '{h2_tag.text}' and '{next_h2_tag.text}':\n{text_between_h2s}\n")
      data.append([title, url, h2_tag.text, cleanhtml(text_between_h2s)])

  # Handle the last h2 tag
  last_h2_tag = h2_tags[-1]
  elements_after_last_h2 = last_h2_tag.find_all_next(string=True)
  text_after_last_h2 = ""
  for element in elements_after_last_h2:
      if element.parent.name == 'p':
          # Get the text content of the <p> tag, including the text within <a> tags
          for content in element.parent.contents:
              if isinstance(content, str):
                  text_after_last_h2 += content
              elif content.name == 'a':
                  text_after_last_h2 += content.text
          text_after_last_h2 += "\n"  # Add a newline after each <p>

  # print(f"Text after '{last_h2_tag.text}':\n{text_after_last_h2}\n")
  data.append([title, url, last_h2_tag.text, cleanhtml(text_after_last_h2)])

x = pages.apply(get_subs, axis=1)
# df = pd.DataFrame(data, columns=['title', 'url', 'heading', 'subheading', 'txt'])
df = pd.DataFrame(data, columns=['title', 'url', 'heading', 'txt'])
print(df.shape)
pprint.pp(df.iloc[0]['txt'])
df.head()

(67, 4)
('  Ultranationalism or extreme nationalism is an extreme form of nationalism '
 'in which a country asserts or maintains detrimental hegemony, supremacy, or '
 'other forms of control over other nations (usually through violent coercion) '
 'to pursue its specific interests. Ultranationalist entities have been '
 'associated with the engagement of political violence even during peacetime.  '
 'In ideological terms, scholars such as the British political theorist Roger '
 'Griffin have found that ultranationalism arises from seeing modern '
 'nation-states as living organisms which are directly akin to physical people '
 'because they can decay, grow, and die, and additionally, they can experience '
 'rebirth. In stark, mythological ways, political campaigners have divided '
 'societies into those which are perceived as being degenerately inferior and '
 'those which are perceived as having great cultural destinies. '
 'Ultranationalism has been an aspect of fascism, with histo

,title,url,heading,txt
0,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Intro,Ultranationalism or extreme nationalism is a...
1,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Contents,
2,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Background concepts and broader context,British political theorist Roger Griffin has s...
3,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Historical movements and analysis,American historian Walter Skya has written in ...
4,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,The following political parties have been char...


In [ ]:
df = df[~((df['heading'] == 'Contents') & (df['txt'].isnull() | (df['txt'] == '')))]
df.head()

,title,url,heading,txt
0,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Intro,Ultranationalism or extreme nationalism is a...
2,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Background concepts and broader context,British political theorist Roger Griffin has s...
3,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Historical movements and analysis,American historian Walter Skya has written in ...
4,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,The following political parties have been char...
5,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist organizations,


In [ ]:
df['title'].value_counts()

,count
title,
Religion,15
Apotheosis,15
Fascism,11
State religion,11
Ultranationalism,10


## LDA (Latent Dirichlet Allocation)

In natural language processing, latent Dirichlet allocation (LDA) is a Bayesian network (and, therefore, a generative statistical model) for modeling automatically extracted topics in textual corpora. The LDA is an example of a Bayesian topic model. In this, observations (e.g., words) are collected into documents, and each word's presence is attributable to one of the document's topics. Each document will contain a small number of topics.

Sources:
 * https://en.wikipedia.org/wiki/Latent_Dirichlet_allocation
 * https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html

Natural Language Processing Basics

* https://github.com/gitmystuff/DSChunks/blob/main/Natural%20Language%20Processing.ipynb

More on LDA

Gemini, October 17 2024

Latent Dirichlet Allocation (LDA) is a statistical method used in natural language processing to uncover hidden thematic structures within a collection of documents. In simpler terms, it's a way to discover the main topics that a set of documents is talking about.

Here's a breakdown of the key concepts:

**1. Documents as Mixtures of Topics**

LDA assumes that each document is a mixture of various topics. For example, a news article might be 70% about politics, 20% about economics, and 10% about sports.

**2. Topics as Distributions of Words**

Each topic is represented as a probability distribution over words. Words that are highly probable within a topic are the words that best characterize that topic. For instance, a "politics" topic might have high probabilities for words like "election," "government," "president," etc.

**3. The Generative Process**

LDA imagines a process for generating documents:

   * For each document:
      * Choose a distribution of topics (e.g., 70% politics, 20% economics, 10% sports).
      * For each word in the document:
         * Choose a topic from the document's distribution of topics.
         * Choose a word from the chosen topic's distribution of words.

**4. Learning the Topics**

LDA's main task is to learn these topic distributions from a given set of documents. It uses statistical inference to figure out which words belong to which topics and how much each topic contributes to each document.

**5. Dirichlet Distributions**

LDA uses Dirichlet distributions as priors for the topic distributions. These priors act as a kind of "regularization" that helps prevent overfitting and encourages the model to find meaningful topics.

**In Summary**

LDA is like a powerful "unmixing" tool. It takes a collection of documents, which are like mixed signals, and separates them into their underlying thematic components. This allows us to understand the main themes present in the data and how those themes are distributed across the documents.

**Here's an analogy:**

Imagine you have a bowl of mixed candies. Each candy represents a word, and each handful of candies represents a document. LDA is like sorting those candies into separate piles based on their type (e.g., chocolate, gummy, sour). Each pile represents a topic, and the proportion of each type of candy in a handful tells you the mixture of topics in that document.

**Applications of LDA:**

* **Topic modeling:** Discovering the main topics in a corpus of text data (e.g., news articles, social media posts, scientific papers).
* **Document classification:** Assigning documents to predefined categories based on their topic distributions.
* **Content recommendation:** Recommending documents or articles to users based on their interests, which can be inferred from their reading history.
* **Customer segmentation:** Grouping customers based on their feedback or reviews, allowing businesses to tailor their marketing strategies.




More on TfidfVectorizer

Gemini, October 17 2024

**TF-IDF (Term Frequency-Inverse Document Frequency)**

TF-IDF is a numerical statistic that reflects how important a word is to a document in a collection or corpus. It's calculated as the product of two metrics:

* **Term Frequency (TF):**  This measures how frequently a term appears in a document. A higher TF indicates that the term is more important within that specific document.

* **Inverse Document Frequency (IDF):** This measures how rare a term is across the entire corpus.  Words that appear in many documents have a low IDF, while words that appear in only a few documents have a high IDF. This helps to downweight common words like "the," "a," and "is," which are less informative for understanding the document's content.

**The TF-IDF formula:**

```
TF-IDF(term, document) = TF(term, document) * IDF(term)
```

**Why is TF-IDF useful?**

TF-IDF helps to identify the most distinctive words in each document, effectively highlighting the words that are most relevant to that document's specific topic or meaning. This is crucial for tasks like:

* **Information retrieval:** Ranking documents based on their relevance to a search query.
* **Text mining:** Identifying key themes and concepts in a collection of documents.
* **Document classification:** Categorizing documents based on their content.

**TfidfVectorizer**

TfidfVectorizer is a class in the scikit-learn library (a popular Python machine learning library) that automates the process of calculating TF-IDF features from text data. It takes raw text documents as input and converts them into a matrix of TF-IDF features.

**Here's what TfidfVectorizer does:**

1. **Tokenization:**  It breaks down the text into individual words or tokens.
2. **Count Vectorization:** It counts the occurrences of each word in each document, creating a "bag-of-words" representation.
3. **TF-IDF Transformation:** It calculates the TF-IDF score for each word in each document, generating a matrix where each row represents a document, and each column represents a word.

**Example using scikit-learn:**

```python
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    'This is the first document.',
    'This document is the second document.',
    'And this is the third one.',
    'Is this the first document?',
]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

# Print the TF-IDF matrix
print(tfidf_matrix)
```

**Key advantages of using TfidfVectorizer:**

* **Efficiency:** It handles the entire TF-IDF calculation process, saving you time and effort.
* **Customization:** It offers various parameters to fine-tune the process, such as controlling the minimum and maximum document frequency of words to be included, removing stop words, and applying different tokenization methods.
* **Integration:** It seamlessly integrates with other scikit-learn tools and pipelines for building machine learning models.

In essence, TF-IDF is a powerful technique for quantifying the importance of words in documents, and TfidfVectorizer provides a convenient way to apply this technique in practice.


In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import TfidfVectorizer

results = 10
components = 10
topics = 10

vectorizer = TfidfVectorizer(stop_words='english')
vectors = vectorizer.fit_transform(df['txt'].values.astype('U'))

model = LatentDirichletAllocation(n_components=components)
model.fit(vectors)

topics_dictionary = {}
for index, topic in enumerate(model.components_):
    print(f'Topic {index} top words: {[vectorizer.get_feature_names_out()[i] for i in topic.argsort()[-topics:]]}')
    topics_dictionary[index] = [vectorizer.get_feature_names_out()[i] for i in topic.argsort()[-topics:]]



Topic 0 top words: ['fascist', 'war', 'pieces', 'lully', 'political', 'parties', 'following', 'ultranationalist', 'characterised', 'judaism']
Topic 1 top words: ['regardless', 'consistent', 'apply', 'appropriate', 'argue', 'parallel', 'scholars', 'religion', 'cultures', 'definition']
Topic 2 top words: ['having', 'country', 'religion', 'religious', 'official', 'known', 'sri', 'egypt', 'pharaohs', 'state']
Topic 3 top words: ['beliefs', 'faith', 'worship', 'religious', 'ming', 'reason', 'deified', 'anthropolatry', 'dynasty', 'religion']
Topic 4 top words: ['form', 'greece', 'official', 'established', 'specific', 'sparta', 'culture', 'governments', 'ancient', 'buddhism']
Topic 5 top words: ['word', 'son', 'nature', 'human', 'fascism', 'political', 'druze', 'god', 'man', 'law']
Topic 6 top words: ['emperor', 'roman', 'practice', 'apotheosis', 'religion', 'parties', 'social', 'solution', 'final', 'criticism']
Topic 7 top words: ['italy', 'fascist', 'ultranationalism', 'italian', 'bundle', 

In [ ]:
def get_topics(row):
  return ', '.join([top for top in topics_dictionary[row.topic_idx]])

topic_results = model.transform(vectors)
df['topic_idx'] = topic_results.argmax(axis=1)

df['topics']= df.apply(get_topics, axis=1)
df.head()

,title,url,heading,txt,topic_idx,topics
0,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Intro,Ultranationalism or extreme nationalism is a...,7,"italy, fascist, ultranationalism, italian, bun..."
2,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Background concepts and broader context,British political theorist Roger Griffin has s...,9,"19th, defined, studying, cultural, founded, pr..."
3,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Historical movements and analysis,American historian Walter Skya has written in ...,6,"emperor, roman, practice, apotheosis, religion..."
4,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,The following political parties have been char...,0,"fascist, war, pieces, lully, political, partie..."
5,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist organizations,,0,"fascist, war, pieces, lully, political, partie..."


## SpaCy

* https://spacy.io/
* https://medium.com/analytics-vidhya/text-summarization-using-spacy-ca4867c6b744

In [ ]:
# uncomment to download
import spacy.cli

spacy.cli.download('en_core_web_sm')

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Language Model and Pipelines

en_core_web_sm

* https://www.kdnuggets.com/2021/03/natural-language-processing-pipelines-explained.html
* https://spacy.io/usage/spacy-101
* https://en.wikipedia.org/wiki/Language_model
* https://builtin.com/data-science/beginners-guide-language-models

More on SpaCy and en_core_web_sm

Gemini, October 17 2024

**spaCy: The NLP Library**

Think of spaCy as a powerful toolbox for Natural Language Processing (NLP) tasks. It provides a wide range of functionalities, including:

* **Tokenization:** Breaking down text into individual words or tokens.
* **Part-of-speech (POS) tagging:** Identifying the grammatical role of each word (e.g., noun, verb, adjective).
* **Named Entity Recognition (NER):** Identifying and classifying named entities like people, organizations, and locations.
* **Dependency parsing:** Analyzing the grammatical structure of sentences.
* **Lemmatization:** Reducing words to their base form (e.g., "running" to "run").

**`en_core_web_sm`: The Language Model**

Now, `en_core_web_sm` is a specific **trained language model** for English that you use *with* spaCy. It's like a pre-built engine that powers spaCy's NLP capabilities.

Here's what `en_core_web_sm` provides:

* **Statistical models:** These models have been trained on a large amount of English text data, enabling spaCy to make accurate predictions about things like POS tags, named entities, and syntactic dependencies.
* **Vocabulary:** A comprehensive vocabulary of English words with their associated linguistic properties.
* **Language-specific rules:** Rules and exceptions that are specific to the English language.

**The Connection**

To use spaCy for NLP tasks on English text, you need to load a language model like `en_core_web_sm`. This model provides the necessary data and algorithms for spaCy to process and understand the text.

**Here's how it works in code:**

```python
import spacy

# Load the English language model
nlp = spacy.load("en_core_web_sm")

# Process a text
text = "This is an example sentence."
doc = nlp(text)

# Access linguistic annotations
for token in doc:
    print(token.text, token.pos_, token.dep_)
```

In this code:

* `spacy.load("en_core_web_sm")` loads the `en_core_web_sm` model.
* `nlp(text)` processes the text using the loaded model.
* The code then accesses linguistic annotations like POS tags (`token.pos_`) and dependency labels (`token.dep_`) provided by the model.

**Different Model Sizes**

spaCy offers different sizes of English language models:

* `en_core_web_sm` (small):  A smaller model that's faster but may be less accurate.
* `en_core_web_md` (medium): A balance of speed and accuracy.
* `en_core_web_lg` (large): A larger model that's more accurate but slower.

You choose the model that best suits your needs based on factors like speed, accuracy requirements, and computational resources.

**In summary:**

spaCy is the framework, and `en_core_web_sm` is the engine that makes it work for English text. You need both to perform NLP tasks effectively.


In [ ]:
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
from string import punctuation
from collections import Counter
from heapq import nlargest

nlp = spacy.load('en_core_web_sm')

In [ ]:
# get example text
import textwrap

textwrap.fill(df.iloc[0]['txt'])

'  Ultranationalism or extreme nationalism is an extreme form of\nnationalism in which a country asserts or maintains detrimental\nhegemony, supremacy, or other forms of control over other nations\n(usually through violent coercion) to pursue its specific interests.\nUltranationalist entities have been associated with the engagement of\npolitical violence even during peacetime.  In ideological terms,\nscholars such as the British political theorist Roger Griffin have\nfound that ultranationalism arises from seeing modern nation-states as\nliving organisms which are directly akin to physical people because\nthey can decay, grow, and die, and additionally, they can experience\nrebirth. In stark, mythological ways, political campaigners have\ndivided societies into those which are perceived as being degenerately\ninferior and those which are perceived as having great cultural\ndestinies. Ultranationalism has been an aspect of fascism, with\nhistoric governments such as the regimes of Fasc

In [ ]:
import textwrap
import re

# data = []
# summary_text = ' '.join([re.sub("\[.*?\]", "", txt) for txt in df.txt])
# doc = nlp(summary_text)
summary_text = ' '.join([re.sub("\[.*?\]", "", df.iloc[0]['txt'])])
doc = nlp(summary_text)
keyword = []
stopwords = list(STOP_WORDS)
pos_tag = ['PROPN', 'ADJ', 'NOUN', 'VERB']
for token in doc:
    if(token.text in stopwords or token.text in punctuation):
        continue
    if(token.pos_ in pos_tag):
        keyword.append(token.text)

freq_word = Counter(keyword)
max_freq = Counter(keyword).most_common(1)[0][1]
for word in freq_word.keys():
    freq_word[word] = (freq_word[word]/max_freq)

sent_strength={}
for sent in doc.sents:
    for word in sent:
        if word.text in freq_word.keys():
            if sent in sent_strength.keys():
                sent_strength[sent] += freq_word[word.text]
            else:
                sent_strength[sent] = freq_word[word.text]

    try:
      data.append([sent_strength[sent], str(sent)])
    except:
      pass
    print(sent_strength[sent])
    print(textwrap.fill(str(sent)))
    print()

summary = nlargest(10, sent_strength, key=sent_strength.get)
summary = ' '.join([w.text for w in summary])
print(textwrap.fill(summary, 100))
# df2 = pd.DataFrame(data, columns=['strength', 'txt'])
# df2.sort_values(by=['strength'], ascending=False).head()

8.999999999999996
  Ultranationalism or extreme nationalism is an extreme form of
nationalism in which a country asserts or maintains detrimental
hegemony, supremacy, or other forms of control over other nations
(usually through violent coercion) to pursue its specific interests.

3.3333333333333335
Ultranationalist entities have been associated with the engagement of
political violence even during peacetime.

9.333333333333332
In ideological terms, scholars such as the British political theorist
Roger Griffin have found that ultranationalism arises from seeing
modern nation-states as living organisms which are directly akin to
physical people because they can decay, grow, and die, and
additionally, they can experience rebirth.

6.333333333333332
In stark, mythological ways, political campaigners have divided
societies into those which are perceived as being degenerately
inferior and those which are perceived as having great cultural
destinies.

7.333333333333331
Ultranationalism has b

In [ ]:
summary = nlargest(int(len(sent_strength)/2), sent_strength, key=sent_strength.get)
summary = ' '.join([w.text for w in summary])
summary = ' '.join([re.sub("\[.*?\]", "", summary)])
print(textwrap.fill(summary))

In ideological terms, scholars such as the British political theorist
Roger Griffin have found that ultranationalism arises from seeing
modern nation-states as living organisms which are directly akin to
physical people because they can decay, grow, and die, and
additionally, they can experience rebirth.   Ultranationalism or
extreme nationalism is an extreme form of nationalism in which a
country asserts or maintains detrimental hegemony, supremacy, or other
forms of control over other nations (usually through violent coercion)
to pursue its specific interests. Ultranationalism has been an aspect
of fascism, with historic governments such as the regimes of Fascist
Italy and Nazi Germany building on ultranationalist foundations by
using specific plans for supposed widespread national renewal.   In
stark, mythological ways, political campaigners have divided societies
into those which are perceived as being degenerately inferior and
those which are perceived as having great cultural des

In [ ]:
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
from string import punctuation

nlp = spacy.blank('en')
nlp.add_pipe('sentencizer')

# https://www.educative.io/answers/text-summarization-in-spacy-and-nltk
# df.iloc[0]['txt']
def summarizer(row):
  txt = row['txt']
  text = ' '.join([re.sub('\[.*?\]|"', '', txt)])
  doc = nlp(text)

  word_frequencies = {}
  for token in doc:
      if token.text not in STOP_WORDS and token.text not in punctuation:
          if token.text not in word_frequencies:
              word_frequencies[token.text] = 1
          else:
              word_frequencies[token.text] += 1


  sorted_sentences = sorted(doc.sents, key=lambda sent: sum(word_frequencies[token.text]
                          for token in sent if token.text in word_frequencies), reverse=True)

  # return str(' '.join(sent.text for sent in sorted_sentences[:int(len(sorted_sentences)/4)]).strip())
  return str(' '.join(sent.text for sent in sorted_sentences[:2]).strip())

# print(textwrap.fill(summarizer(df.iloc[0]['txt'])))

In [ ]:
df['summary']= df.apply(summarizer, axis=1)
df.head()

,title,url,heading,txt,topic_idx,topics,summary
0,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Intro,Ultranationalism or extreme nationalism is a...,7,"italy, fascist, ultranationalism, italian, bun...","In ideological terms, scholars such as the Bri..."
2,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Background concepts and broader context,British political theorist Roger Griffin has s...,9,"19th, defined, studying, cultural, founded, pr...",British political theorist Roger Griffin has s...
3,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Historical movements and analysis,American historian Walter Skya has written in ...,6,"emperor, roman, practice, apotheosis, religion...","By the early twentieth century, fanaticism ari..."
4,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,The following political parties have been char...,0,"fascist, war, pieces, lully, political, partie...",The following political parties have been char...
5,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist organizations,,0,"fascist, war, pieces, lully, political, partie...",


In [ ]:
print(textwrap.fill(df.iloc[4].summary))

In [ ]:
print(textwrap.fill(summarizer(df.iloc[0])))

In ideological terms, scholars such as the British political theorist
Roger Griffin have found that ultranationalism arises from seeing
modern nation-states as living organisms which are directly akin to
physical people because they can decay, grow, and die, and
additionally, they can experience rebirth.   Ultranationalism or
extreme nationalism is an extreme form of nationalism in which a
country asserts or maintains detrimental hegemony, supremacy, or other
forms of control over other nations (usually through violent coercion)
to pursue its specific interests.


# Wikinator II

## Text Emdbedding

* https://huggingface.co/blog/getting-started-with-embeddings

In [ ]:
from google.colab import userdata

model_id = 'sentence-transformers/all-MiniLM-L6-v2'

api_embedding = f'https://api-inference.huggingface.co/pipeline/feature-extraction/{model_id}'
headers = {'Authorization': f'Bearer {userdata.get("HF_TOKEN")}'}

def get_embeddings(row):
  texts = str(row.summary)
  response = requests.post(api_embedding, headers=headers, json={"inputs": texts, "options":{"wait_for_model":True}})
  return response.json()


In [ ]:
df['embeddings']= df.apply(get_embeddings, axis=1)
print(df.shape)
print(df.info())
df.head()

(62, 8)
<class 'pandas.core.frame.DataFrame'>
Index: 62 entries, 0 to 66
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   title       62 non-null     object
 1   url         62 non-null     object
 2   heading     62 non-null     object
 3   txt         62 non-null     object
 4   topic_idx   62 non-null     int64 
 5   topics      62 non-null     object
 6   summary     62 non-null     object
 7   embeddings  62 non-null     object
dtypes: int64(1), object(7)
memory usage: 4.4+ KB
None


,title,url,heading,txt,topic_idx,topics,summary,embeddings
0,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Intro,Ultranationalism or extreme nationalism is a...,7,"italy, fascist, ultranationalism, italian, bun...","In ideological terms, scholars such as the Bri...","[0.049876853823661804, -0.07030076533555984, -..."
2,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Background concepts and broader context,British political theorist Roger Griffin has s...,9,"19th, defined, studying, cultural, founded, pr...",British political theorist Roger Griffin has s...,"[0.061240747570991516, -0.06748878210783005, -..."
3,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Historical movements and analysis,American historian Walter Skya has written in ...,6,"emperor, roman, practice, apotheosis, religion...","By the early twentieth century, fanaticism ari...","[0.03032548353075981, 0.060465116053819656, -0..."
4,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,The following political parties have been char...,0,"fascist, war, pieces, lully, political, partie...",The following political parties have been char...,"[0.04229498654603958, -0.11315956711769104, -0..."
5,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist organizations,,0,"fascist, war, pieces, lully, political, partie...",,"[-0.1188383623957634, 0.048298683017492294, -0..."


In [ ]:
import requests
import numpy as np
from transformers import pipeline, AutoTokenizer
from google.colab import userdata

model_id = 'sentence-transformers/all-MiniLM-L6-v2'

gen = pipeline("text-generation", max_length=500)

# api_embedding = f'https://api-inference.huggingface.co/pipeline/feature-extraction/{model_id}'
api_embedding = pipeline('feature-extraction', model=model_id, tokenizer=AutoTokenizer.from_pretrained(model_id))
# api_gen = 'https://api-inference.huggingface.co/models/gpt2'
headers = {'Authorization': f'Bearer {userdata.get("HF_TOKEN")}'}

def wrap(x):
  return textwrap.fill(x, replace_whitespace=False, fix_sentence_endings=True)

def vector_similarity(vec1, vec2):
  try:
    return np.dot(np.array(vec1), np.array(vec2))
  except:
    pass

def query(payload):
    response = requests.post(api_gen, headers=headers, json=payload)
    return response.json()

def embed(texts):
    embeddings = api_embedding(texts)
    print(embeddings[0][0])
    return embeddings[0][0]

def ask_me_something():
  # question = input('Ask me something: ')
  question = 'What is the connection between ultranationalism and religion and what countries have implemented ultranational and religious policies'
  embedded_prompt = embed(question)
  df['similarity'] = df['embeddings'].apply(lambda vector: vector_similarity(vector, embedded_prompt))
  context = ' '.join(df.nlargest(3, 'similarity')['summary'])
  prompt = f'''
    Only answer the question below if you have 100% certainty of the facts.
    Context: {context}
    Q: {question}
    A:
    '''
  out = gen(prompt, max_length=500)
  print(wrap(out[0]['generated_text']))

ask_me_something()


No model was supplied, defaulted to openai-community/gpt2 and revision 6c0e608 (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_t

[0.3098130226135254, -0.1667851060628891, 0.05676637962460518, -0.1982724517583847, 0.07103373855352402, 0.005247528199106455, -0.1476222574710846, -0.5358078479766846, 0.2848903238773346, -0.1617092341184616, -0.009669189341366291, 0.36686739325523376, -0.16377060115337372, 0.2710549831390381, 0.034326229244470596, -0.12688174843788147, -0.35319051146507263, 0.19626708328723907, 0.0703883096575737, 0.2471742480993271, -0.08765748143196106, -0.4330669045448303, 0.08715890347957611, 0.2877388596534729, 0.23615366220474243, -0.09919721633195877, 0.2272660732269287, -0.026041129603981972, 0.05158061906695366, -0.6852542757987976, -0.07503784447908401, -0.21538591384887695, -0.3494137227535248, -0.17831511795520782, -0.1155465766787529, 0.23743745684623718, -0.04583633318543434, 0.0058632721193134785, 0.17373576760292053, 0.025837039574980736, 0.31313449144363403, -0.18919983506202698, 0.07632755488157272, -0.24223852157592773, 0.13946586847305298, -0.05407554656267166, -0.2738049626350403

## API-Inference Models

* https://huggingface.co/docs/api-inference/en/index

In [ ]:
import requests
import numpy as np
from transformers import pipeline
from google.colab import userdata

# microsoft/phi-1_5
# gpt2
# mistralai/Mistral-Nemo-Instruct-2407
# meta-llama/Llama-3.2-3B-Instruct

api_embedding = f'https://api-inference.huggingface.co/pipeline/feature-extraction/sentence-transformers/all-MiniLM-L6-v2'
api_gen = 'https://api-inference.huggingface.co/models/mistralai/Mistral-Nemo-Instruct-2407'
headers = {'Authorization': f'Bearer {userdata.get("HF_TOKEN")}'}

def wrap(x):
  return textwrap.fill(x, replace_whitespace=False, fix_sentence_endings=True)

def vector_similarity(vec1, vec2):
  try:
    return np.dot(np.array(vec1), np.array(vec2))
  except:
    pass

def query(payload):
    response = requests.post(api_gen, headers=headers, json=payload)
    return response.json()

def embed(texts):
    response = requests.post(api_embedding, headers=headers, json={"inputs": texts, "options":{"wait_for_model":True}})
    return response.json()

def ask_me_something():
  # question = input('Ask me something: ')
  question = 'What is the connection between ultranationalism and religion and what countries have implemented ultranational and religious policies'
  embedded_prompt = embed(question)
  df['similarity'] = df['embeddings'].apply(lambda vector: vector_similarity(vector, embedded_prompt))
  context = ' '.join(df.nlargest(2, 'similarity')['summary'])

  prompt = f'''
    Context: {context}
    Q: {question}
    A:
    '''

  data = query(
      {
          "inputs": prompt,
          "parameters": {"max_length": 1000},
          "use_cache": False,
          "wait_for_model": True
      }
  )

  # print(wrap(data[0]['generated_text']))
  print(data)
  return data

response = ask_me_something()


[{'generated_text': '\n    Context: The degree to which an official national religion is imposed upon citizens by the state in contemporary society varies considerably; from high as in Saudi Arabia and Iran, to none at all as in Greenland, Denmark, England, Iceland, and Greece (in Europe, the state religion might be called in English, the established church). State religions are official or government-sanctioned establishments of a religion, but the state does not need to be under the control of the clergy (as in a theocracy), nor is the state-sanctioned religion necessarily under the control of the state. In ideological terms, scholars such as the British political theorist Roger Griffin have found that ultranationalism arises from seeing modern nation-states as living organisms which are directly akin to physical people because they can decay, grow, and die, and additionally, they can experience rebirth.   Ultranationalism or extreme nationalism is an extreme form of nationalism in w

In [ ]:
print(wrap(response[0]['generated_text']))


    Context: The degree to which an official national religion is
imposed upon citizens by the state in contemporary society varies
considerably; from high as in Saudi Arabia and Iran, to none at all as
in Greenland, Denmark, England, Iceland, and Greece (in Europe, the
state religion might be called in English, the established church).
State religions are official or government-sanctioned establishments
of a religion, but the state does not need to be under the control of
the clergy (as in a theocracy), nor is the state-sanctioned religion
necessarily under the control of the state.  In ideological terms,
scholars such as the British political theorist Roger Griffin have
found that ultranationalism arises from seeing modern nation-states as
living organisms which are directly akin to physical people because
they can decay, grow, and die, and additionally, they can experience
rebirth.   Ultranationalism or extreme nationalism is an extreme form
of nationalism in which a country assert